In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 投影承载上的状态与时钟比较

保留投影margin=1、4组×16通道×4×4支持。新编码为密钥/消息驱动四相位状态，第一次Wan解码前写入；不修改生成Flow。

同一视频和观察比较 global_matched、global_state、local_state。后者含状态预测/创新/更新及最多一次局部时钟变化，不预设优于基线。

选择GPU运行时，从首格依次执行：复用原共享终端，3次VAE解码、36次重编码、0次Transformer。九条视频含正常、匹配重存、删除138、重复138，结果与失败保存Drive。

CPU与结构已检查，真实模型未执行。此前消息模板成功不等于新状态编码成功。AISB暂不接入，尚无共享观测仿射模型证据。


## 拉取固定源码与安装

源码 SHA：`8522a6fa8e2220d3e782931da2c87de68c346916`。复用已跑通的依赖、加载与子进程路径。

In [ ]:
from pathlib import Path
import sys,subprocess
URL='https://github.com/RICHAAARC/SC-SSTW.git'; REF='8522a6fa8e2220d3e782931da2c87de68c346916'; SOURCE=Path('/content/wan_state_clock_source')
if SOURCE.exists():
    if subprocess.check_output(['git','-C',str(SOURCE),'remote','get-url','origin'],text=True).strip()!=URL: raise RuntimeError('unexpected origin')
else:
    subprocess.run(['git','init',str(SOURCE)],check=True); subprocess.run(['git','-C',str(SOURCE),'remote','add','origin',URL],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',REF],check=True); subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','--force','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','diffusers','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True); subprocess.run(['ffmpeg','-version'],check=True)
print('Pinned source:', subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip())


## 运行并持续落盘

输入终端：`/content/drive/MyDrive/Video-WM/C2T1/c2t1_20260915T092031Z/shared_terminal_normalized.pt`。保留源码 config 中的固定候选；默认直接执行。每个运行使用独立目录，日志写在目录同级。

In [ ]:
from datetime import datetime,timezone
CONFIG=SOURCE/'runtime/tstwv2/state_clock_validation.json'; RUN_ID='wan_state_clock_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); OUTPUT=Path('/content/drive/MyDrive/Video-WM/WanStateClock')/RUN_ID
if OUTPUT.exists(): raise FileExistsError(OUTPUT)
import os,signal
cmd=[sys.executable,'-u','-m','runtime.tstwv2.run','--config',str(CONFIG),'--output',str(OUTPUT)]; LOG=OUTPUT.parent/f'{RUN_ID}.launcher.log'; LOG.parent.mkdir(parents=True,exist_ok=True)
with LOG.open('w') as log:
    p=subprocess.Popen(cmd,cwd=SOURCE,start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout: print(line,end=''); log.write(line); log.flush()
        code=p.wait()
    except BaseException:
        try: p.send_signal(signal.SIGTERM)
        except ProcessLookupError: pass
        try: p.wait(timeout=5)
        except subprocess.TimeoutExpired: os.killpg(p.pid,signal.SIGKILL); p.wait()
        raise
print('launcher exit', code)
print('Results:', OUTPUT)
print('Log:', LOG)
if code: raise subprocess.CalledProcessError(code,cmd)


## 查看消息与有效时间对应

检查三种算法消息分差、并列、nominal_time_matching_windows和best_observer。正常时间分母11；删除/重复窗口8仅从时间指标排除，仍参与全部评分。窗口对应不能证明精确定位第138帧。

完整候选及类表在detections；码本、状态驱动、写后记录、MP4和36条观察保存在运行目录。


In [ ]:
import json
result=json.loads((OUTPUT/'result.json').read_text())
print(json.dumps({k:result[k] for k in ('status','source_commit','diagnostic_denominator','fixed_calls','actual_calls','failures') if k in result},ensure_ascii=False,indent=2))
for name,row in result['videos'].items():
    print(name,row['status'])
    for mode,ranking in row.get('detection',{}).get('rankings',{}).items():
        print(mode,'best=',ranking['best'],'message_unique=',ranking['message_unique'])
    print('reporting_only=',row.get('reporting_only'))
print('Full candidates/classes and observer traces:',OUTPUT/'detections')
